<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: DuckDB + the gated HF warehouse.
# Prereqs (per the assignment card):
#  1. Request access on https://huggingface.co/datasets/FlyRank/internship-warehouse (instant approval)
#  2. Create a plain READ token in HF settings (fine-grained tokens can 403 on gated files
#     unless "gated repositories" permission is ticked)
#  3. In Colab: key icon (Secrets) -> add secret named HF_TOKEN -> paste the token -> toggle "Notebook access" on
#     Never paste the token directly into a cell -- this repo is public.

!pip install duckdb --quiet

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

# The four warehouse tables (from the dataset card) -- grain noted for reference,
# but Section 1 below asks YOU to state and then verify your own lane's grain.
#   dim_clients                      104 rows        one row per pseudonymized client
#   dim_content                      519,606 rows     one row per pseudonymized content item
#   fact_content_daily_performance   78,835,655 rows  one row per (report date, client, content item)
#                                     partitioned by month=YYYY-MM
#   fact_content_query_90d           2,414,248 rows   one row per (client, content item, query hash),
#                                     fixed 90-day window (last-30 / prev-30 sub-windows)

MID_PANEL_MONTH = "2026-03"   # iterate here; the _sample file is June 2026 -- sealed test month, per the warning

print("Connected. WAREHOUSE =", WAREHOUSE, "| MID_PANEL_MONTH =", MID_PANEL_MONTH)


Connected. WAREHOUSE = hf://datasets/FlyRank/internship-warehouse | MID_PANEL_MONTH = 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Pull real column names + types before you bucket anything.
for table in ["dim_clients", "dim_content", "fact_content_daily_performance", "fact_content_query_90d"]:
    print(f"--- {table} ---")
    try:
        display(con.sql(f"DESCRIBE SELECT * FROM read_parquet(\'{WAREHOUSE}/{table}/**/*.parquet\') LIMIT 1").df())
    except Exception as e:
        print("  (adjust the glob path above if this errors -- check the exact file layout under Files in the HF repo)", e)
    print()

--- dim_clients ---
  (adjust the glob path above if this errors -- check the exact file layout under Files in the HF repo) HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_clients' (HTTP 404)

--- dim_content ---
  (adjust the glob path above if this errors -- check the exact file layout under Files in the HF repo) HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_content' (HTTP 404)

--- fact_content_daily_performance ---


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



--- fact_content_query_90d ---
  (adjust the glob path above if this errors -- check the exact file layout under Files in the HF repo) HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/fact_content_query_90d' (HTTP 404)



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query 1: GRAIN CHECK ---
# Replace <key columns> with the columns you claimed make one row unique in Section 1.
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 20
""").df()
print(f"Rows where the claimed key repeats: {len(grain_check)}")
grain_check
# If this is empty, your Section 1 grain claim holds. If not, your "one row = one X" is wrong -- fix Section 1.
# --- Query 2: ROW COUNT + DATE SPAN for your slice, month=2026-03 ---
splice_stats = con.sql(f"""
    SELECT
        COUNT(*)                AS n_rows,
        MIN(report_date)      AS earliest_date,
        MAX(report_date)      AS latest_date,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
splice_stats
# --- Query 3: AVAILABILITY, filtered with IS TRUE ---
# Swap <flag_column> for whatever boolean/availability column your DESCRIBE output surfaced
# (e.g. a GSC-indexed flag, or an "is_active" style column). IS TRUE also drops NULLs, which
# is the point -- NULL is not the same as False here.
before = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()["n"][0]

after = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()["n"][0]

print(f"Before filter: {before:,} rows")
print(f"After  IS TRUE filter: {after:,} rows  ({after/before:.1%})")
# --- Feature frame (edit columns to match your Section 2 bucket) ---
feature_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id, report_date,
        gsc_avg_position,
        ai_other,
        ai_copilot,
        ai_claude,
        gsc_data_available
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 100000
""").df()
feature_frame.head()
# --- Deliberate leakage experiment ---

feature_cols = ["gsc_avg_position", "ai_other", "ai_copilot", "ai_claude"]
original_label_col = "gsc_data_available"

df_working = feature_frame[feature_cols + [original_label_col]].dropna().copy()

# Check if the label column has more than one unique class
if df_working[original_label_col].nunique() < 2:
    print(f"\nWarning: The original label column '{original_label_col}' in df_working has only one unique class ({df_working[original_label_col].unique()[0]}).")
    print("Logistic Regression requires at least two classes. Creating a synthetic label for demonstration.")
    # Create a synthetic binary label based on gsc_avg_position median for demonstration
    median_gsc_avg_position = df_working['gsc_avg_position'].median()
    df_working['synthetic_label'] = (df_working['gsc_avg_position'] > median_gsc_avg_position).astype(int)
    y = df_working['synthetic_label']
    label_col = 'synthetic_label' # Update label_col for subsequent steps
    print(f"Using 'synthetic_label' (based on gsc_avg_position > {median_gsc_avg_position}) as the new label.")
    print("Value counts for 'synthetic_label':")
    print(df_working[label_col].value_counts())
else:
    y = df_working[original_label_col]
    label_col = original_label_col

X_honest = df_working[feature_cols]

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.2, random_state=42)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

# Now deliberately add a label-derived column
df_working["LEAK_column"] = df_working[label_col] * np.random.uniform(0.9, 1.1, len(df_working))  # replace with a real leaky derivation
X_leaky = df_working[feature_cols + ["LEAK_column"]]

X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leaky_auc = roc_auc_score(y_te, leaky_model.predict_proba(X_te)[:, 1])
print(f"Leaky AUC (with LEAK_column): {leaky_auc:.3f}  <- should jump toward 1.0")

# Delete the leak and keep the honest number
df_working = df_working.drop(columns=["LEAK_column"])
print(f"\nKeeping the honest number: AUC = {honest_auc:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where the claimed key repeats: 0
Before filter: 9,841,378 rows
After  IS TRUE filter: 3,611,061 rows  (36.7%)

Logistic Regression requires at least two classes. Creating a synthetic label for demonstration.
Using 'synthetic_label' (based on gsc_avg_position > 9.6) as the new label.
Value counts for 'synthetic_label':
synthetic_label
0    10967
1    10959
Name: count, dtype: int64
Honest AUC (no leak): 1.000
Leaky AUC (with LEAK_column): 1.000  <- should jump toward 1.0

Keeping the honest number: AUC = 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify the panel imbalance for your slice -- don't just assert it, show it.
coverage = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{WAREHOUSE}/dim_clients.parquet')
""").df()
coverage.describe(include="all")

,client_hash_id,gsc_data_start,ga4_data_start
count,104,67,51
unique,104,NaN,NaN
top,client_04660893ae39614a,NaN,NaN
freq,1,NaN,NaN
mean,NaN,2025-11-17 00:42:59.104477,2026-02-23 05:38:49.411764
min,NaN,2025-01-27 00:00:00,2025-10-29 00:00:00
25%,NaN,2025-09-24 00:00:00,2026-02-19 00:00:00
50%,NaN,2025-11-05 00:00:00,2026-02-20 00:00:00
75%,NaN,2026-02-19 00:00:00,2026-03-21 12:00:00
max,NaN,2026-06-02 00:00:00,2026-06-01 00:00:00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.